# Simple Linear Regression (SLR)
**Input:** I tại V = 0.65V (µA)  
**Output:** Nồng độ Glucose (mM)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_predict, LeaveOneOut
from scipy.stats import pearsonr
print('✅ Libraries loaded')

## ⚙️ Cấu hình

In [ ]:
FILE_PATH = 'CUCOMOF.csv'
TARGET_V  = 0.65          # Điện thế muốn lấy (V)

## 1. Load Data & Lấy I tại V mục tiêu

In [ ]:
CONCENTRATIONS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
CONC_COLS = [f'{c}mM' for c in CONCENTRATIONS]

data_rows = []
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line or i in [0, 1, 2]:
            continue
        parts = [p.strip() for p in line.split('\t')]
        clean = [p for p in parts if p != '']
        clean = clean[:11] if len(clean) >= 11 else clean + [np.nan]*(11-len(clean))
        data_rows.append(clean)

df_raw = pd.DataFrame(data_rows, columns=['V'] + CONC_COLS)
df_raw = df_raw.apply(pd.to_numeric, errors='coerce')
df_fwd = df_raw.iloc[:df_raw['V'].idxmax()+1].copy().reset_index(drop=True)

# Lấy hàng gần TARGET_V nhất
idx = (df_fwd['V'] - TARGET_V).abs().idxmin()
actual_V = df_fwd.loc[idx, 'V']
I_values = df_fwd.loc[idx, CONC_COLS].values.astype(float)

X = I_values.reshape(-1, 1)     # (10, 1)
y = np.array(CONCENTRATIONS)    # (10,)

print(f'Target V = {TARGET_V}V  →  Actual V trong data = {actual_V:.5f}V')
print(f'\nDữ liệu:')
df_show = pd.DataFrame({'Glucose (mM)': y, f'I @ {actual_V:.4f}V (µA)': I_values})
print(df_show.to_string(index=False))

## 2. Train SLR & Đánh giá

In [ ]:
lr = LinearRegression()
lr.fit(X, y)

y_train = lr.predict(X).ravel()
y_loo   = cross_val_predict(lr, X, y, cv=LeaveOneOut()).ravel()

r2_train   = r2_score(y, y_train)
rmse_train = np.sqrt(mean_squared_error(y, y_train))
r2_loo     = r2_score(y, y_loo)
rmse_loo   = np.sqrt(mean_squared_error(y, y_loo))
r_pearson, p_val = pearsonr(I_values, y)

print('=' * 48)
print(f'  SLR:  Glucose = {lr.coef_[0]:.6f} × I  +  {lr.intercept_:.4f}')
print('=' * 48)
print(f'  Pearson r  = {r_pearson:.4f}   (p = {p_val:.5f})')
print(f'  Train  R²  = {r2_train:.4f}   RMSE = {rmse_train:.4f} mM')
print(f'  LOO-CV R²  = {r2_loo:.4f}   RMSE = {rmse_loo:.4f} mM')
print('=' * 48)

## 3. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Plot 1: Calibration curve I vs [Glucose] ---
ax1 = axes[0]
I_line = np.linspace(I_values.min(), I_values.max(), 200).reshape(-1,1)
y_line = lr.predict(I_line)
ax1.scatter(I_values, y, s=90, color='steelblue', zorder=5, label='Data')
ax1.plot(I_line, y_line, 'r-', linewidth=2, label='SLR fit')
for I_i, y_i in zip(I_values, y):
    ax1.annotate(f'{y_i}mM', (I_i, y_i), textcoords='offset points', xytext=(5, 4), fontsize=8)
ax1.set_xlabel(f'Current I @ {actual_V:.3f}V (µA)', fontsize=11)
ax1.set_ylabel('Glucose (mM)', fontsize=11)
ax1.set_title(f'Calibration Curve\ny = {lr.coef_[0]:.5f}·I + {lr.intercept_:.3f}\nR²={r2_train:.4f}', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

# --- Plot 2: Predicted vs Actual ---
ax2 = axes[1]
ax2.scatter(y, y_loo, s=90, color='darkorange', zorder=5)
lim = [y.min()-0.4, y.max()+0.4]
ax2.plot(lim, lim, 'r--', linewidth=1.5, label='Ideal')
for a, p in zip(y, y_loo):
    ax2.annotate(f'{a}mM', (a, p), textcoords='offset points', xytext=(5, 4), fontsize=8)
ax2.set_xlabel('Actual (mM)', fontsize=11)
ax2.set_ylabel('Predicted (mM)', fontsize=11)
ax2.set_title(f'Predicted vs Actual (LOO-CV)\nR²={r2_loo:.4f}, RMSE={rmse_loo:.4f} mM', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

# --- Plot 3: Residuals ---
ax3 = axes[2]
residuals = y_loo - y
bar_colors = ['#e74c3c' if r < 0 else '#3498db' for r in residuals]
ax3.bar(y, residuals, width=0.08, color=bar_colors, alpha=0.85, edgecolor='black')
ax3.axhline(0, color='black', linewidth=1)
ax3.set_xlabel('Actual (mM)', fontsize=11)
ax3.set_ylabel('Residual (mM)', fontsize=11)
ax3.set_title('Residuals (LOO-CV)', fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'SLR  |  Input: I @ V={actual_V:.3f}V  |  R²(LOO)={r2_loo:.4f}  RMSE={rmse_loo:.4f}mM',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('slr_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Bảng kết quả

In [ ]:
results = pd.DataFrame({
    'Glucose actual (mM)' : y,
    f'I @ {actual_V:.3f}V (µA)' : I_values.round(4),
    'Predicted LOO (mM)'  : y_loo.round(4),
    'Residual (mM)'       : (y_loo - y).round(4),
    'Abs Error (mM)'      : np.abs(y_loo - y).round(4)
})
print(results.to_string(index=False))
print(f'\nMean Abs Error: {results["Abs Error (mM)"].mean():.4f} mM')